# Neuro-Symbolic for Security

## Neuro-Symbolic for Security

Security detection benefits from combining neural anomaly detection with symbolic rule-based systems:

- **Neural component**: Detects subtle behavioral anomalies, handles high-dimensional telemetry
- **Symbolic component**: Enforces known attack signatures, regulatory compliance rules, expert knowledge
- **Hybrid decision**: Alert only when both components agree, or use priority logic

This architecture reduces false positives (symbolic filter) while maintaining the neural component's ability to detect novel threats.

In [ ]:
import numpy as np
from typing import Dict, List, Tuple

np.random.seed(42)

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

class NeuralAnomalyDetector:
    def __init__(self):
        self.mu = None
        self.cov_inv = None
    
    def fit(self, X_normal):
        self.mu = X_normal.mean(0)
        cov = np.cov(X_normal.T) + np.eye(X_normal.shape[1]) * 0.1
        self.cov_inv = np.linalg.inv(cov)
    
    def score(self, x):
        diff = x - self.mu
        return float(diff @ self.cov_inv @ diff)
    
    def predict(self, X, threshold_pct=95):
        scores = np.array([self.score(x) for x in X])
        thresh = np.percentile(scores, threshold_pct)
        return (scores > thresh).astype(int)

class SymbolicRuleEngine:
    def __init__(self):
        self.rules = []
    
    def add_rule(self, name, condition):
        self.rules.append({'name': name, 'condition': condition})
    
    def evaluate(self, event: Dict) -> Tuple[bool, List[str]]:
        triggered = []
        for rule in self.rules:
            try:
                if rule['condition'](event):
                    triggered.append(rule['name'])
            except: pass
        return len(triggered) > 0, triggered

class HybridDetector:
    def __init__(self, neural: NeuralAnomalyDetector, symbolic: SymbolicRuleEngine):
        self.neural = neural
        self.symbolic = symbolic
    
    def detect(self, features: np.ndarray, event: Dict, mode='both'):
        neural_score = self.neural.score(features)
        neural_alert = neural_score > 10.0  # threshold
        symbolic_alert, rules_triggered = self.symbolic.evaluate(event)
        
        if mode == 'both':
            alert = neural_alert and symbolic_alert
        elif mode == 'either':
            alert = neural_alert or symbolic_alert
        else:
            alert = neural_alert
        
        return {'alert': alert, 'neural_score': neural_score,
                'rules_triggered': rules_triggered, 'mode': mode}

# Setup
detector = NeuralAnomalyDetector()
X_normal = np.random.randn(200, 5)
detector.fit(X_normal)

rules = SymbolicRuleEngine()
rules.add_rule('high_priv_after_hours', lambda e: e.get('privilege_level', 0) > 3 and e.get('hour', 12) < 6)
rules.add_rule('mass_deletion', lambda e: e.get('files_deleted', 0) > 100)
rules.add_rule('new_admin_user', lambda e: e.get('new_admin', False) and e.get('source', '') == 'external')

hybrid = HybridDetector(detector, rules)

test_events = [
    (np.random.randn(5), {'privilege_level': 1, 'hour': 10, 'files_deleted': 2, 'new_admin': False}),
    (np.random.randn(5) * 5, {'privilege_level': 5, 'hour': 2, 'files_deleted': 5, 'new_admin': False}),
    (np.random.randn(5), {'privilege_level': 1, 'hour': 3, 'files_deleted': 500, 'new_admin': False}),
    (np.random.randn(5) * 4, {'privilege_level': 4, 'hour': 1, 'files_deleted': 200, 'new_admin': True, 'source': 'external'}),
]

print('Neuro-Symbolic Security Detection:')
for i, (feats, event) in enumerate(test_events):
    r = hybrid.detect(feats, event, mode='either')
    print(f'  Event {i}: alert={r["alert"]}, '
          f'neural={r["neural_score"]:.2f}, rules={r["rules_triggered"]}')
